In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

#EDA

In [ ]:
df_products = pd.read_csv(r'C:\Users\shanm\Downloads\product_info.csv\product_info.csv')

In [ ]:
df_products.shape
df_products.info()

In [ ]:
df_products.isna().sum(axis=0)

In [ ]:
df_products.groupby(['brand_name'])['product_name'].count().sort_values(ascending=False).head(10)

In [ ]:
#popular product_category in each brand

tmp = pd.DataFrame(df_products.groupby(['brand_name','primary_category'])['product_id'].count().reset_index())
tmp['product_rank'] = tmp.groupby(['brand_name'])['product_id'].rank(method = 'min',ascending = False)
tmp[tmp.product_rank == 1]

In [ ]:
#popular product in each brand
tmp1 = df_products[['brand_name','product_name','loves_count']]
tmp1['rank']= tmp1.groupby(['brand_name'])['loves_count'].rank(method='min',ascending=False)
tmp1

In [ ]:
#Average price of each product category
tmp = df_products[['primary_category','secondary_category','tertiary_category','price_usd']]
tmp1 = pd.DataFrame(tmp.groupby(['primary_category','secondary_category','tertiary_category'])['price_usd'].min().reset_index())
tmp1['max_price']=pd.DataFrame(tmp.groupby(['primary_category','secondary_category','tertiary_category'])['price_usd'].max().reset_index())['price_usd']
tmp1['mean_price']=pd.DataFrame(tmp.groupby(['primary_category','secondary_category','tertiary_category'])['price_usd'].mean().reset_index())['price_usd']
tmp1.rename(columns={'price_usd':'min_price'})

In [ ]:
df_products.ingredients.str.replace('[', '', regex=False).str.replace(']', '', regex=False).str.split(',',expand=True)

#Sentiment Analysis

In [ ]:
reviews_250 = pd.read_csv(r"C:\Users\shanm\Downloads\reviews_0_250.csv\reviews_0_250.csv",low_memory = False)
# reviews_500 = pd.read_csv('/kaggle/input/sephora-products-and-skincare-reviews/reviews_250_500.csv',low_memory = False)
# reviwes_750 = pd.read_csv('/kaggle/input/sephora-products-and-skincare-reviews/reviews_500_750.csv',low_memory = False)
# reviwes_1000 = pd.read_csv('/kaggle/input/sephora-products-and-skincare-reviews/reviews_750_1000.csv',low_memory = False)
# reviwes_1500 = pd.read_csv('/kaggle/input/sephora-products-and-skincare-reviews/reviews_1000_1500.csv',low_memory = False)


In [ ]:
reviews = pd.concat([reviews_250])##, reviews_500, reviwes_750, reviwes_1000 , reviwes_1500])
reviews = reviews[['review_text' , 'is_recommended' , 'rating']]
reviews

In [ ]:
reviews.rename(columns = {'is_recommended':'label'},inplace=True)
reviews.isna().sum()

In [ ]:
reviews.isnull().sum()
reviews.dropna(inplace=True)

In [ ]:
reviews.label.astype(int)
reviews.info()

In [ ]:
num_neg = pd.DataFrame(reviews.label.value_counts().reset_index()).iloc[1,1]
num_neg

In [ ]:
reviews_sample = reviews.groupby('label').apply(lambda x : x.sample(num_neg))
reviews_sample.label.value_counts()

#text pre-processing

In [ ]:
from sklearn.utils import shuffle
import string
string.punctuation

In [ ]:
reviews_sample= shuffle(reviews_sample)
reviews_sample.reset_index(drop=True, inplace= True)
reviews_sample

In [ ]:
#reviews_sample.to_csv('/kaggle/working/reviews_sample.csv')

In [ ]:
#reviews_sample = pd.read_csv('/kaggle/working/reviews_sample.csv')

In [ ]:
#number of words in each review
reviews_sample['text_word_count']=reviews_sample['review_text'].apply(lambda x:len(x.split()))
reviews_sample['text_word_count']

To prepare the text data for the model building we perform text preprocessing. It is the very first step of NLP projects. Some of the preprocessing steps are:

Removing punctuations \
Removing URLs\
Removing Stop words\
Lower casing\
Tokenization\
Stemming\
Lemmatization

In [ ]:
# removing punctuations
def remove_punc(text):
    
    clean_text = "".join([i for i in text if i not in string.punctuation])
    return clean_text

reviews_sample['clean_text']  = reviews_sample['review_text'].apply(lambda x: remove_punc(x))


In [ ]:
pip install bs4

In [ ]:
# removing HTML text
from bs4 import BeautifulSoup
def strip_html(text):
    soup = BeautifulSoup(text, "html.parser")
    return soup.get_text()
reviews_sample['clean_text']  = reviews_sample['clean_text'].apply(lambda x: strip_html(x))


In [ ]:
# converting to lower case
reviews_sample['clean_text']  = reviews_sample['clean_text'].apply(lambda x: x.lower())

In [ ]:
import re
def remove_special_characters(text, remove_digits=True):
    text = re.sub("\d", "", text)
    return text
reviews_sample['clean_text']  = reviews_sample['clean_text'].apply(remove_special_characters)

In [ ]:
import nltk
nltk.download('punkt')

In [ ]:
#stop words removal
import nltk 
stopwords = nltk.corpus.stopwords.words('english')
from nltk.tokenize import word_tokenize 
stopwords[0:10]
def remove_stopwords(text):
    text = [i for i in word_tokenize(text) if i not in stopwords]
    return text
reviews_sample['tokenized_text']  = reviews_sample['clean_text'].apply(remove_stopwords)

In [ ]:
#stemming
from nltk.stem.porter import PorterStemmer
porterstemmer = PorterStemmer()
def stemming(text):
    text =[porterstemmer.stem(word) for word in text]
    return text
reviews_sample['tokenized_text']  = reviews_sample['tokenized_text'].apply(stemming)

In [ ]:
reviews_sample[['tokenized_text','clean_text']].iloc[0,:]

In [ ]:
reviews_sample['tokenized_text_2']  = reviews_sample['tokenized_text'].apply(lambda x :' '.join(i for i in x))

In [ ]:
reviews_sample[['tokenized_text_2','clean_text']].iloc[0,:]

In [ ]:
X= reviews_sample['tokenized_text_2']
y = reviews_sample['label'] 

In [ ]:
reviews_sample.to_csv(r'C:\Users\shanm\OneDrive\Desktop\Git_data\Sephora_sample\X_reviews_sample.csv')

In [ ]:
from sklearn.model_selection import train_test_split

# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.50)

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder() # encoding
y_train_label= le.fit_transform(y_train)
y_test_label= le.transform(y_test)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
vec = CountVectorizer(stop_words="english", analyzer='word', 
                            ngram_range=(1, 1), max_df=1.0, min_df=1, max_features=20000)
vec.fit(X_train)
x_train=vec.transform(X_train)
x_test=vec.transform(X_test)

In [ ]:
vec.transform(['happi hear work other hand oili complexion']).toarray()

In [ ]:
vec.get_feature_names_out()[999]

In [ ]:
find =vec.get_feature_names_out()
np.where(find=='favour')

In [ ]:
 x_train = x_train.toarray()

In [ ]:
x_train.shape, y_train.shape

In [ ]:
import tensorflow as tf

In [ ]:
#y_train = tf.keras.utils.to_categorical(y_train)
#y_test = tf.keras.utils.to_categorical(y_test)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
model = Sequential()
model.add(Dense(16, input_dim=x_train.shape[1], activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='Adam',metrics=['accuracy'])
model.summary()

In [ ]:
model.fit(x_train,y_train_label,epochs=10)

In [ ]:
from sklearn.model_selection import train_test_split

# Split the data into train and test sets
X_val, X_test2, y_val, y_test2 = train_test_split(X, y ,test_size=0.80, shuffle = True)

In [ ]:
y_val

In [ ]:
x_val=vec.transform(X_val).toarray()
y_val =le.transform(y_val)

In [ ]:
x_val.shape

In [ ]:
y_pred = model.predict(x_val)

In [ ]:

condlist = [(y_pred>=0.5), (y_pred<0.5)]
choicelist = [1, 0]

y_classes = np.select(condlist, choicelist, default=y_pred)

In [ ]:
y_classes

In [ ]:
from sklearn.metrics import *
confusion_matrix(y_val,y_classes)

In [ ]:
print(classification_report(y_val,y_classes))

In [ ]:
Done